In [1]:
"""
stage1_correlograms.py -- Stage 1: nonparametric cross-correlograms (notebook version).
Empirical JMA-pivot rate at lag L after each stream's pivots / baseline rate.

RULES
  S1.1  For events of a (stream, class) at bar e, hit at lag L <=> is_target[e+L].
        ratio(L) = (hits/valid) / baseline. L = 0..LMAX; L=0 is a co-fire
        diagnostic only (Stage 2 kernels start at L=1). SELF at L=0 is trivially 1.
  S1.2  Within-session only (padding, no wrap). Baseline = target rate over
        non-warm bars. Events are already warm-filtered by Stage 0.
  S1.3  Exogenous streams split by opposing: 'opp' (+1) / 'conf' (-1);
        opposing == 0 dropped. MNQ_JMA_SELF unsplit ('all').
  S1.4  SE via leave-one-session-out jackknife on ratio(L).
  S1.5  Suggested support per curve: max L >= 1 with |z| >= 2 that has a
        significant neighbour (run of >= 2). Feed into Stage 2 bin_edges.
"""

import json
import numpy as np
import pandas as pd

WARMUP_BARS = 10 # from the chart
LAG_SUPPORTS = [1, 2, 3, 5, 8, 13, 21, 34]

FRAME=6
OUT_DIR='data/stage-1'

In [7]:
def _class_list(ev):
    out = []
    for s in sorted(ev["stream"].unique()):
        if s == "MNQ_JMA_SELF":
            out.append((s, "all"))
        else:
            out.append((s, "opp"))
            out.append((s, "conf"))
    return out

def run_stage1(bars, events, out_dir=None, frame=6, lmax=34, plot=True):
    bars = bars.sort_values("bar_index").reset_index(drop=True)
    ev = events[events["stream"] != "TICK_HMA"]
    ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]      # S1.3
    classes = _class_list(ev)
    L = lmax
    lags = np.arange(L + 1)

    S = bars["date"].nunique()
    H = {c: np.zeros((S, L + 1), np.int64) for c in classes}
    N = {c: np.zeros((S, L + 1), np.int64) for c in classes}
    TB = np.zeros(S, np.int64)
    NB = np.zeros(S, np.int64)

    evg = dict(tuple(ev.groupby("date")))
    for si, (sess, g) in enumerate(bars.groupby("date", sort=True)):
        n = len(g)
        first = int(g["bar_index"].iloc[0])
        tgt = g["is_target"].to_numpy()
        warm = g["warm"].to_numpy()
        TB[si] = int(tgt[~warm].sum())
        NB[si] = int((~warm).sum())
        e = evg.get(sess)
        if e is None:
            continue
        tgt_pad = np.concatenate([tgt.astype(np.int64), np.zeros(L + 1, np.int64)])
        for c in classes:
            s, cl = c
            if s == "MNQ_JMA_SELF":
                sub = e[e["stream"] == s]
            else:
                want = 1 if cl == "opp" else -1
                sub = e[(e["stream"] == s) & (e["opposing"] == want)]
            if len(sub) == 0:
                continue
            loc = sub["event_bar"].to_numpy() - first
            idx = loc[:, None] + lags[None, :]                                # S1.2
            H[c][si] = tgt_pad[idx].sum(axis=0)
            N[c][si] = (idx < n).sum(axis=0)

    TBt, NBt = TB.sum(), NB.sum()
    base = TBt / NBt
    rows = []
    supports = {}
    for c in classes:
        Ht, Nt = H[c].sum(0), N[c].sum(0)
        with np.errstate(divide="ignore", invalid="ignore"):
            rate = Ht / Nt
            ratio = rate / base
            H_loo = (Ht[None, :] - H[c]).astype(np.float64)                   # S1.4
            N_loo = (Nt[None, :] - N[c]).astype(np.float64)
            b_loo = (TBt - TB) / (NBt - NB)
            R_loo = (H_loo / N_loo) / b_loo[:, None]
        m = np.isfinite(R_loo)
        k = m.sum(0)
        mean_loo = np.where(k > 0, np.nansum(np.where(m, R_loo, 0.0), 0) / np.maximum(k, 1), np.nan)
        var = (k - 1) / np.maximum(k, 1) * np.nansum(np.where(m, (R_loo - mean_loo) ** 2, 0.0), 0)
        se = np.sqrt(var)
        z = (ratio - 1.0) / se
        name = f"{c[0]}|{c[1]}"
        for l in range(L + 1):
            rows.append((name, c[0], c[1], l, int(Nt[l]), int(Ht[l]),
                         float(rate[l]), float(base), float(ratio[l]),
                         float(se[l]), float(z[l])))
        sig = np.zeros(L + 1, bool)
        fz = np.isfinite(z)
        sig[fz] = np.abs(z[fz]) >= 2.0
        best = 0
        for l in range(1, L + 1):                                             # S1.5
            if sig[l] and (sig[l - 1] or (l + 1 <= L and sig[l + 1])):
                best = l
        supports[name] = best

    df = pd.DataFrame(rows, columns=["key", "stream", "cls", "lag", "n_valid",
                                     "hits", "rate", "baseline", "ratio", "se", "z"])
    summary = {"baseline_rate": base, "n_sessions": int(S),
               "suggested_support_bars": supports}

    if out_dir is not None:
        df.to_csv(f"{out_dir}/correlograms_{frame}s.csv", index=False)
        with open(f"{out_dir}/stage1_summary_{frame}s.json", "w") as f:
            json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))

    if plot:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        keys = sorted(df["key"].unique())
        nc = 3
        nr = (len(keys) + nc - 1) // nc
        fig = make_subplots(rows=nr, cols=nc, subplot_titles=keys)
        for i, kk in enumerate(keys):
            r, cix = i // nc + 1, i % nc + 1
            d = df[df["key"] == kk]
            up = d["ratio"] + 2 * d["se"]
            dn = d["ratio"] - 2 * d["se"]
            fig.add_trace(go.Scatter(x=d["lag"], y=up, line=dict(width=0),
                                     showlegend=False, hoverinfo="skip"), row=r, col=cix)
            fig.add_trace(go.Scatter(x=d["lag"], y=dn, line=dict(width=0), fill="tonexty",
                                     fillcolor="rgba(100,100,200,0.25)",
                                     showlegend=False, hoverinfo="skip"), row=r, col=cix)
            fig.add_trace(go.Scatter(x=d["lag"], y=d["ratio"], mode="lines",
                                     line=dict(color="#1f4e9c"), showlegend=False), row=r, col=cix)
            fig.add_trace(go.Scatter(x=[0, L], y=[1, 1], mode="lines",
                                     line=dict(color="black", dash="dot", width=1),
                                     showlegend=False), row=r, col=cix)
        fig.update_layout(height=320 * nr, width=1400,
                          title=f"Stage 1 correlograms, {frame}s (ratio to baseline, +/-2 SE jackknife)")
        if out_dir is not None:
            fig.write_html(f"{out_dir}/correlograms_{frame}s.html")
        fig.show()

    return df, summary

In [8]:
bars = pd.read_parquet(f'data/stage-0/bars_{FRAME}s.parquet')
events = pd.read_parquet(f'data/stage-0/events_{FRAME}s.parquet')

df, summary = run_stage1(bars, events, out_dir=OUT_DIR, frame=6, lmax=34, plot=True)


{
  "baseline_rate": 0.10302256633210928,
  "n_sessions": 1163,
  "suggested_support_bars": {
    "MNQ_D1|opp": 7,
    "MNQ_D1|conf": 11,
    "MNQ_D2|opp": 31,
    "MNQ_D2|conf": 8,
    "MNQ_JMA_SELF|all": 10,
    "TICK_D1|opp": 7,
    "TICK_D1|conf": 10,
    "TICK_D2|opp": 7,
    "TICK_D2|conf": 9
  }
}
